# D1-R: SIM-CoT GPT-2 as a mixture of experts at half the FFN

Every criterion is frozen in `results/ffn_sparsity/d1r/PLAN.md`, and the notebook refuses to start unless it matches
`PLAN.sha256`.

1. **Checks:**
   - the dense evaluation must repeat step selection's answers;
   - the untrained pruned model must repeat SPARSE-0's S50;
   - 51 dense training updates must repeat Stage 1b's `ft6_step` losses.
2. **Conversion, about 2 minutes:** each FFN becomes 768 shared neurons plus 12 experts of 192, with a 2-layer MLP router,
   built from SPARSE-0's calibration file on Drive.
3. **Six fine-tunes:** MoE-50, Prune-50 and Hash-50 at seeds 7 and 8, with Stage 1b's `ft6_step` recipe. MoE at seed 7
   goes first; if it scores 192 or less, the run stops there.
4. **Verdict:** does MoE-50 match the dense `ft6_step` (209 and 200), and does it beat pruning and a hash router?

**Before running:** Runtime → Change runtime type → **G4**, then Runtime → **Run all**. It takes about 40 minutes. If Colab
disconnects, Run all again resumes from the last finished arm; one resume is allowed.

In [ ]:
REPO_REF = "main"         # the branch or commit holding scripts/ffn_sparsity and the frozen plan

In [ ]:
from pathlib import Path
import hashlib, json, os, runpy, shutil, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/dynamic-state-anchoring")
BACKUP_DIR = DRIVE / "d1r"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode:
    raise SystemExit("No GPU: Runtime > Change runtime type > GPU, then Runtime > Run all.")
print("GPU:", gpu.stdout.strip())

PINNED = BACKUP_DIR / "commit.txt"  # a started run keeps the code it started with
ref = PINNED.read_text().strip() if PINNED.exists() else REPO_REF
REPO_DIR = Path("/content/dynamic-state-anchoring")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/sttruji/dynamic-state-anchoring.git", str(REPO_DIR)],
                   check=True)
subprocess.run(["git", "fetch", "origin", ref], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
plan = Path("results/ffn_sparsity/d1r/PLAN.md")
pinned = Path("results/ffn_sparsity/d1r/PLAN.sha256").read_text().split()[0]
if hashlib.sha256(plan.read_bytes()).hexdigest() != pinned:
    raise SystemExit("PLAN.md does not match PLAN.sha256: the frozen plan was changed; stop.")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
PINNED.write_text(COMMIT)
sessions = BACKUP_DIR / "sessions.txt"
count = int(sessions.read_text()) + 1 if sessions.exists() else 1
if count > 2:
    raise SystemExit("D1-R allows one session and one resume; this would be a third.")
sessions.write_text(str(count))
print("Code:", COMMIT, f"(from {ref}); plan sha256 matches; session {count} of at most 2")

In [ ]:
VENV = Path("/content/dsa-venv")
subprocess.run([sys.executable, "scripts/setup_colab.py", "--venv-dir", str(VENV),
                "--requirements", "requirements-colab.txt"], check=True)
PYTHON = str(VENV / "bin/python")
os.environ["DSA_DATA_DIR"] = str(DRIVE / "data")
os.environ["HF_HOME"] = "/content/hf-cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
run_logged = runpy.run_path("scripts/colab_process.py")["run_logged"]

def run(*args):
    run_logged([PYTHON, *map(str, args)], cwd=REPO_DIR, log_dir=BACKUP_DIR / "logs")

TRAIN = DRIVE / "data" / "gsm_train.json"  # the GSM8K-Aug training file the inner-loop runs used
assert TRAIN.exists(), f"{TRAIN} is missing; it is the training file from the inner-loop notebook's runs"
VALID = DRIVE / "data" / "bench" / "eval" / "gsm8k_valid.jsonl"  # the locked validation file step selection used
assert VALID.exists(), f"{VALID} is missing; run scripts/setup_data.py as the step selection notebook did"
CALIBRATION = DRIVE / "ffn-sparsity" / "run" / "calibration.pt"  # SPARSE-0's, pinned by the plan's sha256
assert CALIBRATION.exists(), f"{CALIBRATION} is missing; it is SPARSE-0's calibration file"
CHECKPOINT = DRIVE / "checkpoints/simcot-gpt2"
run("scripts/latent_gate/fetch_simcot.py", "--output", CHECKPOINT)

In [ ]:
SHA = "2dd1d19ec3ddb1099177ac801c88f40364031c9cf882aebf1b195cf0d263b75f"  # SPARSE-0's calibration file
RUN, LOCAL = BACKUP_DIR / "run", Path("/content/d1r")  # results on Drive; trained weights stay on the runtime
EVALS, CONVERSION = RUN / "evals", RUN / "conversion.pt"

def evaluate(name, checkpoint, arm, *extra):
    if (EVALS / f"{name}.jsonl").exists():
        print(f"{name}: already evaluated")
        return
    conversion = [] if arm == "dense" else ["--conversion", CONVERSION]
    run("scripts/ffn_sparsity/eval_moe.py", "--checkpoint", checkpoint, "--arm", arm, *conversion,
        "--output", EVALS / f"{name}.jsonl", "--device", "cuda", *extra)

def right(name):
    return sum(json.loads(x)["correct"] for x in (EVALS / f"{name}.jsonl").read_text().splitlines())

def fine_tune(arm, seed):
    name, out = f"{arm}-s{seed}", LOCAL / f"{arm}-s{seed}"
    if (EVALS / f"{name}.jsonl").exists():
        print(f"{name}: already done ({right(name)})")
        return
    if not (out / "model/pytorch_model.bin").exists():
        run("scripts/ffn_sparsity/train_moe.py", "--arm", arm, "--conversion", CONVERSION, "--checkpoint", CHECKPOINT,
            "--output", out, "--train-file", TRAIN, "--seed", seed, "--device", "cuda", "--bf16")
    keep = RUN / "training" / name
    keep.mkdir(parents=True, exist_ok=True)
    for file in ("metrics.jsonl", "args.json"):
        shutil.copy2(out / file, keep / file)
    evaluate(name, out / "model", arm)
    print(f"{name}: {right(name)} of 500")

# R0a, the evaluation path
evaluate("base-dense", CHECKPOINT, "dense", "--reference", "results/step_select/stage0/evals/seed1.jsonl")
# The conversion
if not CONVERSION.exists():
    run("scripts/ffn_sparsity/convert.py", "--checkpoint", CHECKPOINT, "--calibration", CALIBRATION, "--sha256", SHA,
        "--output", CONVERSION, "--train-file", TRAIN, "--device", "cuda")
# R0b, the conversion; then the other training-free arms, reported only
evaluate("prune-0", CHECKPOINT, "prune", "--limit", 100, "--reference", "results/ffn_sparsity/run/evals/S50.jsonl")
evaluate("moe-0", CHECKPOINT, "moe", "--limit", 100)
evaluate("hash-0", CHECKPOINT, "hash", "--limit", 100)
# R0c, the training path: 51 dense updates against Stage 1b's ft6_step
check = RUN / "training/dense-s7-check"
if not (check / "check.json").exists():
    shutil.rmtree(LOCAL / "dense-s7-check", ignore_errors=True)
    run("scripts/ffn_sparsity/train_moe.py", "--arm", "dense", "--checkpoint", CHECKPOINT,
        "--output", LOCAL / "dense-s7-check", "--train-file", TRAIN, "--seed", 7, "--device", "cuda", "--bf16",
        "--stop-after", 51, "--check", "results/inner_loop/stage1b/training/ft6_step/metrics.jsonl")
    check.mkdir(parents=True, exist_ok=True)
    for file in ("check.json", "metrics.jsonl", "args.json"):
        shutil.copy2(LOCAL / "dense-s7-check" / file, check / file)
print("R0c:", json.loads((check / "check.json").read_text()))

In [ ]:
fine_tune("moe", 7)
if right("moe-s7") <= 192:
    print(f"Early kill: MoE-50 at seed 7 scored {right('moe-s7')} <= 192, so the run stops here.")
else:
    for arm, seed in (("prune", 7), ("hash", 7), ("moe", 8), ("prune", 8), ("hash", 8)):
        fine_tune(arm, seed)

In [ ]:
report = subprocess.run([PYTHON, "scripts/ffn_sparsity/report_d1r.py", str(RUN)], cwd=REPO_DIR,
                        capture_output=True, text=True)
print(report.stdout, report.stderr)
(BACKUP_DIR / "report.txt").write_text(report.stdout)

In [ ]:
# The evaluations, training logs, checks and conversion summary, for the repo's results folder. The conversion itself
# (conversion.pt) stays on Drive at d1r/run.
from google.colab import files

bundle = Path("/content/d1r-results")
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree(RUN, bundle / "run", ignore=shutil.ignore_patterns("*.pt", "*.tmp"))
for name in ("report.txt", "commit.txt", "sessions.txt"):
    shutil.copy2(BACKUP_DIR / name, bundle / name)
files.download(shutil.make_archive(str(bundle), "zip", bundle))
print(f"Downloading {bundle}.zip. If it does not start, use the Files panel on the left.")

**Reading the result**
- **R0a, R0b and R0c** must all pass, or there is no verdict.
- **Fine-tuned, per seed** gives each arm's score next to `ft6_step`'s 209 and 200.
- **Pooled** gives each arm against the dense control, and MoE against Prune and against Hash.
- **The last line** is the verdict.

**Sharing:** the last cell downloads a zip. Point Claude at it.